##### How to filter Data Using Year, Month, Day?
- The **date** column must be of **DateType or TimestampType** for these methods to work efficiently.
- If your **date** column is a **string**, you must first **convert** it to a **date type** using the **cast(), to_date() or to_timestamp()** functions.

In [0]:
from pyspark.sql.functions import col, lit, year, month, day, dayofmonth, count, sum, concat_ws, current_date

In [0]:
# Create a DataFrame with a date column
data = [(1, "2022-01-01", 100),
        (2, "2022-01-01", 250),
        (3, "2023-01-02", 200),
        (4, "2023-01-02", 250),
        (5, "2023-01-02", 150),
        (6, "2023-01-03", 150),
        (7, "2024-02-01", 300),
        (8, "2024-02-01", 200),
        (9, "2025-05-09", 500),
        (10, "2026-02-02", 150),
        (11, "2026-02-02", 250),
        (12, "2026-03-10", 350)]

sales_df = spark.createDataFrame(data, ["id", "transaction_date", "amount"])

# Convert the date column to a DateType
sales_df = sales_df.withColumn("transaction_date", sales_df["transaction_date"].cast("date"))
display(sales_df)

id,transaction_date,amount
1,2022-01-01,100
2,2022-01-01,250
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150
6,2023-01-03,150
7,2024-02-01,300
8,2024-02-01,200
9,2025-05-09,500
10,2026-02-02,150


##### 1) Filter by Year
- This returns **all records from 2025**.

In [0]:
sales_df.filter(year("transaction_date") == 2023).display()

id,transaction_date,amount
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150
6,2023-01-03,150


##### 2) Filter by Month
- Returns all records from **May (any year)**.

In [0]:
sales_df.filter(month("transaction_date") == 5).display()

id,transaction_date,amount
9,2025-05-09,500


##### 3) Filter by Day
- Returns records where **day = 2**.

In [0]:
sales_df.filter(dayofmonth("transaction_date") == 2).display()

id,transaction_date,amount
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150
10,2026-02-02,150
11,2026-02-02,250


##### 4) Filter by Year + Month
- Returns **January 2023** data.

In [0]:
sales_df.filter(
    (year("transaction_date") == 2023) &
    (month("transaction_date") == 1)
).display()

id,transaction_date,amount
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150
6,2023-01-03,150


##### 5) Filter by a specific year, month and day
- Returns data for **2 January 2023**.

In [0]:
# Filter for a specific date (e.g., January 2, 2023)
sales_df.filter(
    (year("transaction_date") == 2023) &
    (month("transaction_date") == 1) &
    (dayofmonth("transaction_date") == 2)
).display()

id,transaction_date,amount
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150


##### 6) Filter using direct comparison with a date string (Filter based on a specific date)
- PySpark can implicitly **cast** a **standard yyyy-MM-dd string** to a **date type** during comparison, which is often more efficient.

In [0]:
# Filter for all dates after January 1, 2020
filtered_df = sales_df.filter(col('transaction_date') > lit('2025-01-01').cast('date'))
display(filtered_df)

id,transaction_date,amount
9,2025-05-09,500
10,2026-02-02,150
11,2026-02-02,250
12,2026-03-10,350


##### 7) Best Practice
- Instead of separating **year/month/day**, many data engineers use **date comparison**.

In [0]:
sales_df.filter(col("transaction_date") == "2023-01-02").display()

id,transaction_date,amount
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150


##### 8) How to filter based on a date range?

**Method 01:**

     # specify start and end dates
     dates = ('2023-01-01', '2025-12-31')

     # filter DataFrame to only show rows between start and end dates
     df.filter(df.start_date.between(*dates)).display()

     # count number of rows in DataFrame that fall between start and end dates
     df.filter(df.start_date.between(*dates)).count()

**Method 02:**

     df.filter(col("transaction_date").between("2023-01-01", "2025-12-31")).display()

**Method 03:**

     startDate = "2022-01-01"
     endDate = "2022-01-31"
     filteredDf = df.filter(to_date($"date_col", "yyyy-MM-dd").between(startDate, endDate))

In [0]:
sales_df.filter(
    col("transaction_date").between("2023-01-01", "2025-12-31")
).display()

id,transaction_date,amount
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150
6,2023-01-03,150
7,2024-02-01,300
8,2024-02-01,200
9,2025-05-09,500


##### 9) How to filter based on the current date?

In [0]:
sales_df.filter(col("transaction_date") == current_date()).display()

id,transaction_date,amount
12,2026-03-10,350


##### 10) Filter based on a date difference
- rows where the **difference** between the **transaction_date and the current date** is **greater than 30 days**.

In [0]:
from pyspark.sql.functions import datediff

final_sales_fltr = sales_df \
  .select("*", datediff(current_date(), col("transaction_date")).alias("days_diff")) \

display(final_sales_fltr)

id,transaction_date,amount,days_diff
1,2022-01-01,100,1529
2,2022-01-01,250,1529
3,2023-01-02,200,1163
4,2023-01-02,250,1163
5,2023-01-02,150,1163
6,2023-01-03,150,1162
7,2024-02-01,300,768
8,2024-02-01,200,768
9,2025-05-09,500,305
10,2026-02-02,150,36


In [0]:
final_sales_fltr \
  .filter(datediff(current_date(), col("transaction_date")) > 1000) \
  .display()

id,transaction_date,amount,days_diff
1,2022-01-01,100,1529
2,2022-01-01,250,1529
3,2023-01-02,200,1163
4,2023-01-02,250,1163
5,2023-01-02,150,1163
6,2023-01-03,150,1162


##### 11) Filter using SQL expression strings

In [0]:
# Filter for a specific year using an SQL expression
sales_df_final = sales_df.filter("YEAR(transaction_date) = 2023")
display(sales_df_final)

id,transaction_date,amount
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150
6,2023-01-03,150


| Filter     | Code                                             |
| ---------- | ------------------------------------------------ |
| Year       | `year(col("date")) == 2025`                      |
| Month      | `month(col("date")) == 3`                        |
| Day        | `dayofmonth(col("date")) == 15`                  |
| Year+Month | `(year(col)==2025) & (month(col)==3)`            |
| Exact Date | `col("date") == "2025-03-15"`                    |
| Date Range | `col("date").between("2025-03-01","2025-03-31")` |